# Lesson 3: Connecting to a SQL Database

## Setup LLM + SQL

Jalankan `python setup_data.py` jika database belum ada.

In [1]:
from llm_setup import get_chat_model, check_llm_config, print_provider_info

from langchain_community.agent_toolkits import create_sql_agent, SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase

issues = check_llm_config()
if issues:
    raise EnvironmentError("\n".join(issues))

print_provider_info()

C:\Users\USER\AppData\Local\Temp\ipykernel_42624\3085918263.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.agent_toolkits import create_sql_agent, SQLDatabaseToolkit


Provider : ollama
Model    : llama3.1:8b
Ollama   : http://localhost:11434


## Recover the original dataset and move to SQL database

In [2]:
from sqlalchemy import create_engine
import pandas as pd

database_file_path = "./db/test.db"

engine = create_engine(f"sqlite:///{database_file_path}")
file_url = "./data/all-states-history.csv"
df = pd.read_csv(file_url).fillna(value=0)
df.to_sql(
    "all_states_history",
    con=engine,
    if_exists="replace",
    index=False,
)

20780

## Prepare the SQL prompt

In [3]:
from prompts import MSSQL_AGENT_PREFIX, MSSQL_AGENT_FORMAT_INSTRUCTIONS

## Call the Azure Chat model and create the SQL agent

In [4]:
from llm_setup import get_agent_type, get_agent_executor_kwargs

llm = get_chat_model(temperature=0, max_tokens=2000)

db = SQLDatabase.from_uri(f"sqlite:///{database_file_path}")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
print(f"Agent type: {get_agent_type()}")

Agent type: zero-shot-react-description


In [5]:
from llm_setup import use_simple_sql

QUESTION = """How may patients were hospitalized during October 2020
in New York, and nationwide as the total of all states?
Use the hospitalizedIncrease column
"""

if use_simple_sql():
    print("Ollama: lewati SQL agent penuh (sering hang). Pakai sql_helper di cell berikutnya.")
else:
    agent_type = get_agent_type()
    create_kwargs = dict(
        llm=llm,
        toolkit=toolkit,
        agent_type=agent_type,
        top_k=30,
        verbose=True,
        max_iterations=15,
        max_execution_time=120,
        agent_executor_kwargs=get_agent_executor_kwargs(),
    )
    if agent_type != "tool-calling":
        create_kwargs["prefix"] = MSSQL_AGENT_PREFIX
        create_kwargs["format_instructions"] = MSSQL_AGENT_FORMAT_INSTRUCTIONS
    agent_executor_SQL = create_sql_agent(**create_kwargs)

Ollama: lewati SQL agent penuh (sering hang). Pakai sql_helper di cell berikutnya.


## Invoke the SQL model

**Ollama:** pakai jalur sederhana (2–5 menit). **Azure/Groq/OpenAI:** pakai SQL agent penuh.

In [6]:
from llm_setup import use_simple_sql
from sql_helper import ask_sql_simple

if use_simple_sql():
    print("Mode Ollama: jalur SQL sederhana (lebih cepat)...")
    result = ask_sql_simple(llm, db, QUESTION)
    print("SQL:", result["sql"])
    print("Result:", result["result"])
    print("\n", result["output"])
else:
    agent_executor_SQL.invoke(QUESTION)

Mode Ollama: jalur SQL sederhana (lebih cepat)...
SQL: SELECT 
    SUM(CASE WHEN state = 'NY' THEN hospitalizedIncrease ELSE 0 END) AS ny_hospitalized,
    SUM(hospitalizedIncrease) AS national_hospitalized
FROM 
    all_states_history
WHERE 
    date LIKE '2020-10%';
Result: [(0, 53485)]

 **Hospitalized Patients in October 2020**

### New York
#### 0 patients were hospitalized during October 2020 in New York.

### Nationwide (all states)
#### 53,485 patients were hospitalized nationwide during October 2020.

**Explanation:**
The SQL query used to retrieve the data is as follows:

```sql
SELECT 
    SUM(CASE WHEN state = 'NY' THEN hospitalizedIncrease ELSE 0 END) AS ny_hospitalized,
    SUM(hospitalizedIncrease) AS national_hospitalized
FROM 
    all_states_history
WHERE 
    date LIKE '2020-10%';
```

This query uses a `CASE` statement to sum up the `hospitalizedIncrease` column for New York (NY) specifically, and another `SUM` aggregation function to calculate the total number of ho